# 🧪 AWS Architecture - Parsimonious 1-Phase Batch Evaluation & Prompt Lab

Este notebook implementa el entorno profesional de **Evaluación de Prompts para el Modelo Parsimonioso en 1 sola Fase** (utilizando la versión de producción **Prompt v9** con Gemini 3.6 Flash).

---
### 1. Ventajas del Enfoque Parsimonioso en 1 Fase:
- **1 sola llamada a la API por video:** Reduce la latencia y consumo de tokens a la mitad.
- **Fidelidad Visual Estricta:** Prioriza los componentes dibujados físicamente en la pizarra.
- **Descomposición Obligatoria de Cajas Agrupadas (Regla v9):** Extrae servicios específicos (`CloudTrail`, `GuardDuty`, `S3`, `SQS`) cuando se mencionan verbalmente en cajas como 'AWS Cloud Logs'.
- **Preservación de Baseline:** Mantiene intacto el Grafo Parsimonioso Original real para comparaciones justas.


## 📌 Paso 1: Importaciones y Configuración del Entorno

In [ ]:
import os
import sys
import base64
import json
import shutil
import re
import time
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from pathlib import Path
from pydantic import BaseModel
from google import genai

# Configuración de rutas del proyecto
CURRENT_DIR = Path.cwd()
if CURRENT_DIR.name == "whiteboard_selection_lab":
    LAB_DIR = CURRENT_DIR
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR
    LAB_DIR = PROJECT_ROOT / "whiteboard_selection_lab"

sys.path.append(str(PROJECT_ROOT))

from config.settings import GEMINI_API_KEY
from scripts.core.graph_builder import create_graph_from_cloudscape_json
from scripts.utils.evaluate_graphs import evaluate_pair, load_services_catalog

print("✓ Entorno Parsimonioso inicializado con éxito.")
print(f"  • Raíz del proyecto: {PROJECT_ROOT}")
print(f"  • Directorio del laboratorio: {LAB_DIR}")
print(f"  • Gemini API Key configurada: {'Sí' if GEMINI_API_KEY else 'No'}")

## ⚙️ Paso 2: Panel de Control del Experimento Parsimonioso

In [ ]:
# ==============================================================================
# ⚙️ CONFIGURACIÓN DE PARÁMETROS
# ==============================================================================

MODEL_NAME = 'gemini-3.6-flash'
EXPERIMENT_LABEL = "Prompt_v9_Parsimonious"
FORCE_RERUN = False

# Selección de Videos a Evaluar (puedes ajustar la lista)
TARGET_VIDEOS = [
    "3WgTBTDlQN8", "-3lnf5lzsH0", "-wLEkq21cvA", "0F7KDLz-kIQ", "1aYoIZvabbk"
]

print("📋 EXPERIMENTO PARSIMONIOSO CONFIGURADO:")
print(f"   • Modelo: {MODEL_NAME}")
print(f"   • Etiqueta: {EXPERIMENT_LABEL}")
print(f"   • Force Re-run: {FORCE_RERUN}")
print(f"   • Cantidad de videos: {len(TARGET_VIDEOS)}")

## 📝 Paso 3: Definición del Prompt v9 Parsimonioso

In [ ]:
services_csv = PROJECT_ROOT / "graph_renderer" / "services.csv"
if not services_csv.exists():
    services_csv = PROJECT_ROOT / "data" / "cloudscape_gt" / "services.csv"

df_services = pd.read_csv(services_csv)
df_actores = df_services[df_services['is_aws'] == False]
df_aws = df_services[df_services['is_aws'] == True]

lista_actores = ", ".join(df_actores['name'].dropna().astype(str).unique())
lista_aws = ", ".join(df_aws['name'].dropna().astype(str).unique())

PROMPT_V9_TEXT = """You are an expert AWS Solutions Architect. You are analyzing a whiteboard screenshot from an AWS "This is My Architecture" YouTube video, along with the full transcript of the video.

Your task is to extract the cloud architecture shown, encoding it using the Cloudscape dataset schema (FAST25 paper by Satija et al.). Since this is a STRICTLY PARSIMONIOUS model, your primary ground truth is the VISUAL whiteboard diagram.

## RULES:
1. STRICT EXACT AWS SERVICES (NO SHORTENING/TRUNCATION): You MUST strictly use the exact string from the <AWS_SERVICES_PLACEHOLDER> list for the `service` field. Shortening, truncating, or abbreviating service names is STRICTLY FORBIDDEN (e.g., if the list says 'KinesisDataStream' or 'ApiGateway', you MUST use the exact string regardless of how the presenter pronounces it or writes it colloquially). This avoids typographic mismatches.

2. EXTERNAL/INTERNAL ACTORS (SEMANTIC ONTOLOGY MAPPING): 
   Identify what comes from "outside" the core AWS architecture based on the visual drawing. You MUST classify these actors by choosing EXCLUSIVELY from the <ACTORS_PLACEHOLDER> list.
   - Use semantic reasoning to match the visual element (and its brief context) to the most precise label available (e.g., matching a drawn physical device to 'UserConsumerIOT' or 'UserConsumerEdge', a hospital to 'UserConsumerHospital', or a business team to 'UserCompanyAnalyst').
   - Default generic internal staff to 'UserCompanyDeveloper' and generic external users to 'UserConsumerWebMobile' if no specific visual/contextual clues are present.
   - External non-AWS technologies (like CouchBase, SAP, ServiceNow, or custom public APIs) should be mapped to `ThirdParty` or their exact Partner name if present in the schema.

3. VISUAL-FIRST NODES & OMIT FLOATING TEXT: Base your nodes primarily on physical boxes or distinct icons drawn with a clear contour. 
   - IGNORE standalone floating text or handwritten explanatory words that do not have a bounding box or icon.

4. MANDATORY DECOMPOSITION OF GROUPED BOXES: 
   - If a box on the whiteboard represents a collection of AWS services (e.g. labeled 'AWS', 'Security Sources', or 'AWS Cloud Logs') AND the transcript or presenter explicitly names the specific AWS services contained within it (such as CloudTrail, GuardDuty, SQS, SNS, S3):
   - You ARE REQUIRED to break down that single box into individual nodes for EACH explicitly named AWS service.
   - DO NOT create a single generic 'ThirdParty' or 'AWS Cloud Logs' node when specific AWS services are explicitly named in the audio/transcript.
   - If an arrow points to the boundary of the container, route connections directly to the decomposed internal service nodes.

5. STRICT VISUAL & ESSENTIAL EDGES (BALANCED CONNECTIONS): 
   - Base your connections primarily on explicit, directional physical line arrows (->) drawn on the whiteboard.
   - Trace round-trip or return connections (<-) ONLY if they have a clear visual representation on the whiteboard (such as double arrowheads or explicit return line drawings) OR if they are indispensable to the primary synchronous execution flow drawn.
   - DO NOT mass-connect external actors or services to all components. Only draw entry and return connections that have a clear visual origin or explicit primary flow path.
   - Avoid generating speculative or decorative return paths that are not backed by visual line indicators.

6. LOGICAL SEQUENCING (FLOW FROM EXTERNAL ACTORS): When assigning `flow_id` (integer) and `seq` (string) to edges, always trace the sequence starting from external actors (UserConsumer*, UserCompany*, ThirdParty) moving progressively inwards toward the backend.

7. PARSIMONY PRINCIPLE (VISUAL DEDUPLICATION): 
   - Keep the graph structurally clean. Deduplicate multiple instances of the SAME service if they perform the exact same logical step.

8. FORMATTING: Edges must have `flow_id` (integer), `seq` (string), and `type` ("data" or "meta", default "data"). The `id` of nodes must be an integer string.

## OUTPUT FORMAT:
Return ONLY valid JSON (no markdown fences):
{
  "step_by_step_reasoning": "Briefly analyze the visual components and explicit arrows...",
  "graph": {
    "name": "<title>", "link": "", "categories": "<category>", "graph_usable": true, "notes": "..."
  },
  "nodes": [ {"id": "0", "service": "...", "name": "", "notes": "..."} ],
  "edges": [ {"source": "0", "target": "1", "flow_id": 0, "seq": "0", "type": "data", "notes": ""} ]
}"""

ACTIVE_PARSIMONIOUS_PROMPT = PROMPT_V9_TEXT.replace("<ACTORS_PLACEHOLDER>", lista_actores).replace("<AWS_SERVICES_PLACEHOLDER>", lista_aws)
print("✓ Prompt v9 Parsimonioso configurado correctamente.")

## 🚀 Paso 4: Ejecución del Pipeline Parsimonioso en Lote

In [ ]:
class GraphMetadata(BaseModel):
    name: str
    link: str
    categories: str
    graph_usable: bool
    notes: str

class Node(BaseModel):
    id: str
    service: str
    name: str
    notes: str

class Edge(BaseModel):
    source: str
    target: str
    flow_id: int
    seq: str
    type: str
    notes: str

class FinalArchitectureSchema(BaseModel):
    step_by_step_reasoning: str
    graph: GraphMetadata
    nodes: list[Node]
    edges: list[Edge]

client = genai.Client(api_key=GEMINI_API_KEY)
catalog_dict = load_services_catalog(services_csv)
results_list = []

for idx, video_id in enumerate(TARGET_VIDEOS, 1):
    print(f"\n[{idx}/{len(TARGET_VIDEOS)}] Procesando Video {video_id}...")
    lab_workspace = LAB_DIR / "lab_workspace" / video_id
    lab_workspace.mkdir(parents=True, exist_ok=True)

    cache_json = lab_workspace / "test_analysis.json"
    cache_graphml = lab_workspace / "test_graph.graphml"
    pars_orig_path = lab_workspace / "test_graph_original.graphml"

    # Preservar la baseline del Parsimonio Original
    if not pars_orig_path.exists():
        cmd = ["git", "show", f"origin/main:data/graphs_parsimonious/{video_id}.graphml"]
        res = subprocess.run(cmd, capture_output=True, text=True)
        if res.returncode == 0 and len(res.stdout) > 50:
            with open(pars_orig_path, "w", encoding="utf-8") as f:
                f.write(res.stdout)

    ws_whiteboard = lab_workspace / "best_whiteboard.jpg"
    ws_transcript = LAB_DIR / "transcriptions" / f"{video_id}_transcript.json"

    src_good = PROJECT_ROOT / "data" / "good_whiteboard" / f"{video_id}.jpg"
    src_rep = PROJECT_ROOT / "cloudscape_reports" / video_id / "best_whiteboard.jpg"
    src_frames = LAB_DIR / "frames_new" / f"{video_id}_pizarra" / "best_whiteboard.jpg"

    if src_good.exists(): shutil.copy2(src_good, ws_whiteboard)
    elif src_rep.exists(): shutil.copy2(src_rep, ws_whiteboard)
    elif src_frames.exists(): shutil.copy2(src_frames, ws_whiteboard)

    src_t = PROJECT_ROOT / "data" / "raw" / f"{video_id}_transcript.json"
    if src_t.exists():
        LAB_DIR.joinpath("transcriptions").mkdir(parents=True, exist_ok=True)
        shutil.copy2(src_t, ws_transcript)

    image_bytes = ws_whiteboard.read_bytes()
    image_b64 = base64.b64encode(image_bytes).decode("utf-8")

    with open(ws_transcript, "r", encoding="utf-8") as f:
        segments = json.load(f)
    transcript_text = " ".join(s.get("text", "").strip() for s in segments)

    full_prompt = f"{ACTIVE_PARSIMONIOUS_PROMPT}\n\n## FULL TRANSCRIPT:\n{transcript_text}"

    res_data = None
    if cache_json.exists() and not FORCE_RERUN:
        print(f"  ✓ Usando análisis en caché para {video_id}")
        with open(cache_json, "r", encoding="utf-8") as f:
            res_data = json.load(f)
    else:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=[
                {
                    "parts": [
                        {"text": full_prompt},
                        {"inline_data": {"mime_type": "image/jpeg", "data": image_b64}}
                    ]
                }
            ],
            config={
                "response_mime_type": "application/json",
                "response_schema": FinalArchitectureSchema
            }
        )
        res_data = json.loads(response.text)
        with open(cache_json, "w", encoding="utf-8") as f:
            json.dump(res_data, f, indent=2, ensure_ascii=False)

    test_graph = create_graph_from_cloudscape_json(res_data)
    nx.write_graphml(test_graph, str(cache_graphml))

    # Evaluación de Métricas
    gt_graph_path = PROJECT_ROOT / "data" / "cloudscape_gt" / f"{video_id}.graphml"
    gt_g = nx.read_graphml(str(gt_graph_path))
    eval_metrics = evaluate_pair(test_graph, gt_g, video_id, catalog_dict)

    results_list.append({
        "video_id": video_id,
        "title": gt_g.graph.get('name', f"Video {video_id}"),
        "svc_f1": eval_metrics['svc_f1'],
        "svc_precision": eval_metrics['svc_precision'],
        "svc_recall": eval_metrics['svc_recall'],
        "edge_f1": eval_metrics['edge_f1'],
        "edge_precision": eval_metrics['edge_precision'],
        "edge_recall": eval_metrics['edge_recall'],
        "nodes_count": test_graph.number_of_nodes(),
        "edges_count": test_graph.number_of_edges(),
        "missing": eval_metrics.get('services_missing', []),
        "hallucinated": eval_metrics.get('services_hallucinated', [])
    })
    print(f"  ✓ Svc F1: {eval_metrics['svc_f1']*100:.1f}% | Edge F1: {eval_metrics['edge_f1']*100:.1f}%")

df_results = pd.DataFrame(results_list)
display(df_results[['video_id', 'svc_f1', 'svc_precision', 'svc_recall', 'edge_f1', 'edge_precision', 'edge_recall']])

## 📊 Paso 5: Visualización de Gráficas y Resumen de Métricas

In [ ]:
sns.set_theme(style="darkgrid")
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.barplot(data=df_results, x='video_id', y='svc_f1', palette='Blues_d')
plt.title('Service F1 por Video (Parsimonioso v9)')
plt.ylim(0, 1.05)
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
sns.barplot(data=df_results, x='video_id', y='edge_f1', palette='Greens_d')
plt.title('Edge F1 por Video (Parsimonioso v9)')
plt.ylim(0, 1.05)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

print(f"⭐ PROMEDIO GLOBAL - Service F1: {df_results['svc_f1'].mean()*100:.1f}% | Edge F1: {df_results['edge_f1'].mean()*100:.1f}%")